In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split

<h2 align="center" style="color:red">Loading the Data</h2>

In [ ]:
df = pd.read_excel("premiums.xlsx")
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.replace(' ', '_').str.lower()
df.columns

<h2 align="center" style="color:red">EDA & Data Cleaning</h2>

### Handling Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
print(df.shape)
df.isna().sum()

### Handling Duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df[df['number_of_dependants'] < 0].shape

In [ ]:
df[df['number_of_dependants'] < 0]['number_of_dependants'].unique()

In [ ]:
# We will assume that these values were actually positive but were entered negative due to some errors
df['number_of_dependants'] = abs(df['number_of_dependants'])

In [ ]:
df['number_of_dependants'].describe()

### Boxplot for numeric columns analysis

#### Analysis of Univariate columns

In [ ]:
numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns
numeric_columns

In [ ]:
for cols in numeric_columns:
    sns.boxplot(x = df[cols])
    plt.show()

In [ ]:
df[df['age'] > 100]['age'].unique()

In [ ]:
df1 = df[df['age'] <= 100].copy()
df1['age'].describe()

In [ ]:
def iqr_bounds(col):
    Q1, Q3 = col.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower_bnd = Q1 - IQR * 1.5
    upper_bnd = Q3 + IQR * 1.5
    return lower_bnd, upper_bnd

In [ ]:
lower, upper = iqr_bounds(df1['income_lakhs'])
lower, upper

In [ ]:
df1[df1['income_lakhs'] > upper].shape

In [ ]:
# Since there are too many people with income above upper_bound, we will change the upper bound
income_threshold = df1['income_lakhs'].quantile(0.999)
income_threshold

In [ ]:
df1[df1['income_lakhs'] > income_threshold].shape

In [ ]:
df2 = df1[df1['income_lakhs'] <= income_threshold].copy()
df2.shape

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))  # Adjust the size as necessary

for i, column in enumerate(numeric_columns):
    ax = axs[i//2, i%2]  # Determines the position of the subplot in the grid
    sns.histplot(df2[column], kde=True, ax=ax)
    ax.set_title(column)

plt.tight_layout()
plt.show()

#### Bivariate Analysis

In [ ]:
numeric_features = ['age', 'income_lakhs', 'number_of_dependants']
fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 6))  # Adjust figure size as necessary

for ax, column in zip(axes, numeric_features):
    sns.scatterplot(x=df2[column], y=df2['annual_premium_amount'], ax=ax)
    ax.set_title(f'{column} vs. Annual Premium Amount')
    ax.set_xlabel(column)
    ax.set_ylabel('Annual Premium Amount')

plt.tight_layout()  # Adjust layout
plt.show()

In [ ]:
categorical_cols = ['gender', 'region', 'marital_status', 'bmi_category', 'smoking_status', 'employment_status', 'income_level', 'medical_history', 'insurance_plan']
for col in categorical_cols:
    print(col , '->' , df2[col].unique())

In [ ]:
df2['smoking_status'].replace({
    'Smoking=0': 'No Smoking',
    'Does Not Smoke': 'No Smoking',
    'Not Smoking': 'No Smoking'
}, inplace=True)
df2['smoking_status'].unique()

In [ ]:
df['gender'].value_counts()

In [ ]:
sns.barplot(x = df2['gender'].unique(), y=df['gender'].value_counts(normalize=True) * 100)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

for ax, column in zip(axes, categorical_cols):
    
    category_counts = df2[column].value_counts(normalize=True) * 100
    
    sns.barplot(x=category_counts.index, y=category_counts.values, ax=ax)
    
    ax.set_title(f'Percentage Distribution of {column}')
    ax.set_ylabel('Percentage of Policyholders (%)')
    ax.set_xlabel(column)
    
    # Rotate x-axis labels
    ax.tick_params(axis='x', rotation=50)

plt.tight_layout()
plt.show()

In [ ]:
crosstab = pd.crosstab(df2['income_level'], df2['insurance_plan'])
print(crosstab)

crosstab.plot(kind='bar', stacked=True)
plt.title('Income vs Plan')
plt.ylabel('Count')
plt.show()

In [ ]:
df2.head(3)

In [ ]:
sns.heatmap(crosstab, annot=True, fmt='d')
plt.title('Income level vs Insurance Plans')
plt.show()

<h2 align="center" style="color:red">Feature Engineering</h2>

In [ ]:
risk_scores = {
    "diabetes": 6,
    "heart disease": 8,
    "high blood pressure":6,
    "thyroid": 5,
    "no disease": 0,
    "none":0
}

df2[['disease1', 'disease2']] = df2['medical_history'].str.split(" & ", expand=True).apply(lambda x: x.str.lower())
df2['disease1'].fillna('none', inplace=True)
df2['disease2'].fillna('none', inplace=True)
df2['total_risk_score'] = 0

for disease in ['disease1', 'disease2']:
    df2['total_risk_score'] += df2[disease].map(risk_scores)

In [ ]:
# we are normalizing the riskscores
max_score = df2['total_risk_score'].max()
min_score = df2['total_risk_score'].min()
df2['normalized_risk_score'] = (df2['total_risk_score'] - min_score) / (max_score - min_score)
df2.head(3)

#### Encoding categorical columns

In [ ]:
df2['insurance_plan'] = df2['insurance_plan'].map({'Bronze': 1, 'Silver': 2, 'Gold': 3})
df2['insurance_plan'].unique()

In [ ]:
df2['income_level'] = df['income_level'].map({'<10L':1, '10L - 25L': 2, '25L - 40L':3, '> 40L':4})
df2['income_level'].unique()

In [ ]:
nominal_cols = ['gender', 'region', 'marital_status', 'bmi_category', 'smoking_status', 'employment_status']
df3 = pd.get_dummies(df2, columns=nominal_cols, drop_first=True, dtype=int)
df3.head(3)

#### Selecting Features

In [ ]:
df4 = df3.drop(['medical_history','disease1', 'disease2', 'total_risk_score'], axis=1)
df4.head(3)                

In [ ]:
cm = df4.corr()

plt.figure(figsize=(20,12))
sns.heatmap(cm, annot= True)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
X  = df4.drop('annual_premium_amount', axis = True)
y = df4['annual_premium_amount']
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
cols_to_scale = ['age','number_of_dependants', 'income_level',  'income_lakhs', 'insurance_plan']
X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

In [ ]:
X.describe()

In [ ]:
X.values

In [ ]:
def calculate_vif(data):
    vif_df = pd.DataFrame()
    vif_df['column'] = data.columns
    vif_df['VIF'] = [variance_inflation_factor(data.values, i) for i in range(data.shape[1])]
    return vif_df


In [ ]:
calculate_vif(X)

In [ ]:
calculate_vif(X.drop('income_level', axis="columns"))

In [ ]:
# we are gonna drop income_lakhs due to high VIF value
X_reduced = X.drop('income_level', axis="columns")

<h2 align="center" style="color:red">Model Training</h2>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.30, random_state=12)

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
test_score = model_lr.score(X_test, y_test)
train_score = model_lr.score(X_train, y_train)
train_score, test_score

In [ ]:
y_pred = model_lr.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)
print("Linear Regression --> MSE: ", mse_lr, "RMSE: ", rmse_lr)

In [ ]:
model_lr.coef_, model_lr.intercept_

In [ ]:
feature_importance = model_lr.coef_
coef_df = pd.DataFrame(feature_importance, index=X_train.columns, columns=['coefs'])
coef_df = coef_df.sort_values(by='coefs', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(coef_df.index, coef_df['coefs'], color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Feature Importance in Linear Regression')
plt.show()

In [ ]:
model_rg = Ridge(alpha=5)
model_rg.fit(X_train, y_train)
test_score = model_rg.score(X_test, y_test)
train_score = model_rg.score(X_train, y_train)
train_score, test_score

In [ ]:
y_pred = model_rg.predict(X_test)

mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)
print("Ridge Regression --> MSE: ", mse_lr, "RMSE: ", rmse_lr)

In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=20, max_depth=3)
model_xgb.fit(X_train, y_train)
model_xgb.score(X_test, y_test)

In [ ]:
y_pred = model_xgb.predict(X_test)

mse_lr = mean_squared_error(y_test, y_pred)
rmse_lr = np.sqrt(mse_lr)
print("XGBoost Regression --> MSE: ", mse_lr, "RMSE: ", rmse_lr)

In [ ]:
model_xgb = XGBRegressor()
param_grid = {
    'n_estimators': [20, 40, 50, 70],
    'learning_rate': [0.01, 0.1, 0.2, 0.4],
    'max_depth': [3, 4, 5, 7],
}
random_search = RandomizedSearchCV(model_xgb, param_grid, n_iter=10, cv = 3, scoring='r2', random_state=15, n_jobs=-1)
random_search.fit(X_train, y_train)
random_search.best_score_

In [ ]:
random_search.best_params_

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
feature_importance = best_model.feature_importances_

# Create a DataFrame for easier handling
coef_df = pd.DataFrame(feature_importance, index=X_train.columns, columns=['Coefficients'])

# Sort the coefficients for better visualization
coef_df = coef_df.sort_values(by='Coefficients', ascending=True)

# Plotting
plt.figure(figsize=(8, 4))
plt.barh(coef_df.index, coef_df['Coefficients'], color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Feature Importance in XGBoost')
plt.show()

<h2 align="center" style="color:red">Error Analysis</h2>

In [ ]:
y_pred = best_model.predict(X_test)

residuals = y_pred - y_test
residuals_pct = (residuals / y_test) * 100

results_df = pd.DataFrame({
    'actual': y_test, 
    'predicted': y_pred, 
    'diff': residuals, 
    'diff_pct': residuals_pct
})
results_df.head()

In [ ]:
sns.histplot(results_df['diff_pct'], kde=True)
plt.title('Distribution of Residuals')
plt.xlabel('Diff PCT')
plt.ylabel('Frequency')
plt.show()

In [ ]:
extreme_error_threshold = 10
extreme_results_df = results_df[np.abs(results_df['diff_pct']) > extreme_error_threshold]
extreme_results_df.head()

In [ ]:
extreme_results_df.shape

In [ ]:
extreme_errors_pct = extreme_results_df.shape[0]*100/X_test.shape[0]
extreme_errors_pct

In [ ]:
extreme_results_df[abs(extreme_results_df.diff_pct)>50].sort_values("diff_pct",ascending=False)

In [ ]:
extreme_errors_df = X_test.loc[extreme_results_df.index]
extreme_errors_df.head(2)

In [ ]:
extreme_errors_df.shape

In [ ]:
for feature in X_test.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(extreme_errors_df[feature], color='blue', label='Extreme Errors', kde=True)
    sns.histplot(X_test[feature], color='indigo', label='Overall', alpha=0.5, kde=True)
    plt.legend()
    plt.title(f'Distribution of {feature} for Extreme Errors vs Overall')
    plt.show()

#### Reverse Scaling

In [ ]:
extreme_errors_df['income_level']=-1
df_reversed = pd.DataFrame()
df_reversed[cols_to_scale] = scaler.inverse_transform(extreme_errors_df[cols_to_scale])
df_reversed.head()

In [ ]:
df_reversed.describe()

In [ ]:
sns.histplot(df_reversed.age)
plt.show()

##### This shows that majority of the extreme errors are coming from young age group of < 25 years of age. We need to may be build a separate model for this segment.